# 预训练数据盘点与列表生成（Drive + GCS）

**流程（拆分）**

1. **阶段一 — Drive**：对每个数据集 **单独运行一格扫描**，写入 `per_dataset/<名>_spatial.txt`；或用「一键全部」循环。
2. **阶段二 — GCS**：先认证，再 **按桶单独运行** BraTS / FLARE23 / RSNA-PED 抽样（可选全量）。
3. **阶段三 — 合并**：汇总 spatial、生成 DeepLesion 语义、BraTS pair4 contrast。

**路径约定**

- Drive：`dataset/pretrain/<数据集>/`，列表根目录 `pretrain_lists_inventory/`。
- GCS：`gs://brats_preprocessed_npy`（contrast 四模态）、`gs://flare23` 与 `gs://rsna_2022_ped_preprocessed_npy/npy2`（spatial）。
- **不含 LIDC**；FLARE23 在 GCS 上为 spatial。

训练时 `np.load` 需本地路径或 gcsfuse；`gs://` 仅作清单占位时请自行替换前缀。

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Cell 2: 依赖
!pip install -q numpy tqdm

In [ ]:
# Cell 3: 路径配置 + 按数据集扫描 / GCS 工具函数
from __future__ import annotations

import json
import os
import random
import subprocess
from collections import defaultdict
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

DRIVE_PRETRAIN = Path("/content/drive/MyDrive/dataset/pretrain")
DRIVE_DATASET = Path("/content/drive/MyDrive/dataset")

LISTS_OUT = DRIVE_PRETRAIN / "pretrain_lists_inventory"
LISTS_OUT.mkdir(parents=True, exist_ok=True)
(LISTS_OUT / "per_dataset").mkdir(exist_ok=True)
(LISTS_OUT / "gcs_samples").mkdir(exist_ok=True)

DRIVE_INVENTORY_JSON = LISTS_OUT / "drive_inventory.json"

DRIVE_DATASET_NAMES = [
    "AbdomenCT-1K",
    "Amos",
    "DeepLesion",
    "FLARE22",
    "FLARE23",
    "ISLES",
    "RibFrac",
    "RSNA-2022-CSFD",
    "STOIC",
    "TCIA",
    "TotalSegmentator",
]

EXTRA_SPATIAL_ROOTS = [
    DRIVE_DATASET / "TCIA_Covid19" / "patch_random_spatial",
]

GCS_DATASETS = {
    "brats_preprocessed_npy": "gs://brats_preprocessed_npy",
    "flare23": "gs://flare23",
    "rsna_2022_ped_npy2": "gs://rsna_2022_ped_preprocessed_npy/npy2",
}

SKIP_DIR_NAMES = {"__MACOSX", ".git"}


def find_under_pretrain(preferred_name: str) -> Path | None:
    if not DRIVE_PRETRAIN.is_dir():
        return None
    exact = DRIVE_PRETRAIN / preferred_name
    if exact.is_dir():
        return exact
    lower = preferred_name.lower()
    for child in DRIVE_PRETRAIN.iterdir():
        if child.is_dir() and child.name.lower() == lower:
            return child
    return None


def npy_roots_for_dataset(ds_root: Path) -> list[Path]:
    tagged = []
    for sub in ("npy", "npy1", "npy2", "npy3", "npy4", "patch_random_spatial"):
        p = ds_root / sub
        if p.is_dir():
            tagged.append(p)
    if tagged:
        return tagged
    return [ds_root] if ds_root.is_dir() else []


def iter_npys_under(root: Path):
    if not root.is_dir():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIR_NAMES]
        if "__MACOSX" in dirpath:
            continue
        for fn in filenames:
            if fn.endswith(".npy"):
                yield Path(dirpath) / fn


def collect_npys_for_roots(roots: list[Path]) -> list[str]:
    out: list[str] = []
    seen = set()
    for r in roots:
        for p in iter_npys_under(r):
            s = str(p)
            if s not in seen:
                seen.add(s)
                out.append(s)
    out.sort()
    return out


def write_lines(path: Path, lines: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def _load_drive_inventory() -> dict:
    data = {"drive_pretrain": str(DRIVE_PRETRAIN), "datasets": {}}
    if DRIVE_INVENTORY_JSON.is_file():
        try:
            data = json.loads(DRIVE_INVENTORY_JSON.read_text(encoding="utf-8"))
        except Exception:
            pass
    if "datasets" not in data:
        data["datasets"] = {}
    return data


def _save_drive_inventory(data: dict) -> None:
    DRIVE_INVENTORY_JSON.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def scan_single_drive_dataset(name: str, update_json: bool = True) -> dict:
    """只扫 pretrain 下一个子文件夹，写入 per_dataset/<name>_spatial.txt。"""
    ds_root = find_under_pretrain(name)
    entry = {"resolved_root": str(ds_root) if ds_root else None, "npy_total": 0, "roots": []}
    out_txt = LISTS_OUT / "per_dataset" / f"{name.replace('/', '_')}_spatial.txt"
    if ds_root is None:
        print(f"[{name}] 未找到目录 -> 写入空列表 {out_txt.name}")
        write_lines(out_txt, [])
        if update_json:
            inv = _load_drive_inventory()
            inv["datasets"][name] = entry
            _save_drive_inventory(inv)
        return entry
    roots = npy_roots_for_dataset(ds_root)
    all_paths: list[str] = []
    for r in roots:
        sub = collect_npys_for_roots([r])
        entry["roots"].append({"path": str(r), "count": len(sub)})
        all_paths.extend(sub)
    all_paths = sorted(set(all_paths))
    entry["npy_total"] = len(all_paths)
    write_lines(out_txt, all_paths)
    print(f"[{name}] npy={len(all_paths)} -> {out_txt}")
    if update_json:
        inv = _load_drive_inventory()
        inv["datasets"][name] = entry
        _save_drive_inventory(inv)
        print("已合并写入", DRIVE_INVENTORY_JSON)
    return entry


def scan_extra_spatial(extra: Path, update_json: bool = True) -> tuple[str, dict]:
    """例如 TCIA_Covid19/patch_random_spatial。"""
    key = f"EXTRA:{extra.name}@{extra.parent.name}"
    paths = collect_npys_for_roots([extra])
    entry = {
        "resolved_root": str(extra),
        "npy_total": len(paths),
        "roots": [{"path": str(extra), "count": len(paths)}],
    }
    fname = f"EXTRA_{extra.parent.name}_{extra.name}_spatial.txt"
    write_lines(LISTS_OUT / "per_dataset" / fname, paths)
    print(f"[{key}] npy={len(paths)} -> per_dataset/{fname}")
    if update_json:
        inv = _load_drive_inventory()
        inv["datasets"][key] = entry
        _save_drive_inventory(inv)
    return key, entry


def gsutil_ls_r_npy(gcs_prefix: str, timeout_sec: int = 7200) -> list[str]:
    cmd = f"gsutil ls -r '{gcs_prefix}/**/*.npy' 2>/dev/null"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout_sec)
    return [x.strip() for x in r.stdout.splitlines() if x.strip().endswith(".npy")]


def gcs_scan_one_bucket(label: str, full_scan: bool = False, sample_cap: int = 50) -> list[str]:
    """写入 gcs_samples/{label}_sample.txt；full_scan 时写 gcs_samples/{label}_all_npy.txt（慎用）。"""
    prefix = GCS_DATASETS[label]
    gcs_out = LISTS_OUT / "gcs_samples"
    print("===", label, prefix, "FULL_SCAN=" + str(full_scan), "===")
    if full_scan:
        files = gsutil_ls_r_npy(prefix.rstrip("/"))
        write_lines(gcs_out / f"{label}_all_npy.txt", files)
        print("全量条数:", len(files), "->", gcs_out / f"{label}_all_npy.txt")
    else:
        quick = subprocess.run(
            f"gsutil ls '{prefix}/**' 2>/dev/null | head -n {sample_cap}",
            shell=True,
            capture_output=True,
            text=True,
            timeout=120,
        )
        files = [x.strip() for x in quick.stdout.splitlines() if x.strip()]
        write_lines(gcs_out / f"{label}_sample.txt", files[:sample_cap])
        print("抽样条数:", len(files), "->", gcs_out / f"{label}_sample.txt")
    partial = LISTS_OUT / "gcs_inventory_partial.json"
    agg = {}
    if partial.is_file():
        try:
            agg = json.loads(partial.read_text(encoding="utf-8"))
        except Exception:
            pass
    agg[label] = {"prefix": prefix, "count": len(files), "sample": files[:20]}
    partial.write_text(json.dumps(agg, indent=2, ensure_ascii=False), encoding="utf-8")
    return files


print("LISTS_OUT =", LISTS_OUT)

## 阶段一 A — Drive：**单个**数据集扫描

修改 `DATASET_NAME` 后运行下一格；对每个数据集重复一次（或跳过没有数据的集）。输出：`per_dataset/<名>_spatial.txt`，并 **增量更新** `drive_inventory.json`。

In [ ]:
# 修改名称后单独运行本格
DATASET_NAME = "AbdomenCT-1K"  # 改成 Amos / DeepLesion / FLARE22 / … 之一

scan_single_drive_dataset(DATASET_NAME)

## 阶段一 B — Drive：**一键**扫描全部（可选）

等价于对 `DRIVE_DATASET_NAMES` 逐个调用；会覆盖对应 `per_dataset/*.txt`。

In [ ]:
if not DRIVE_PRETRAIN.is_dir():
    raise SystemExit("请先挂载 Drive")

inv = {"drive_pretrain": str(DRIVE_PRETRAIN), "datasets": {}}
for name in tqdm(DRIVE_DATASET_NAMES, desc="Drive all"):
    inv["datasets"][name] = scan_single_drive_dataset(name, update_json=False)
_save_drive_inventory(inv)
print("已写入", DRIVE_INVENTORY_JSON)

## 阶段一 C — Drive：**额外** TCIA 旧路径（可选）

对应 `dataset/TCIA_Covid19/patch_random_spatial`，写入 `per_dataset/EXTRA_*_spatial.txt`。

In [ ]:
for extra in EXTRA_SPATIAL_ROOTS:
    if extra.is_dir():
        scan_extra_spatial(extra)
    else:
        print("跳过（不存在）:", extra)

## 阶段二 — GCS：**按桶单独**抽样或全量

先运行 **认证**，再按需运行 **BraTS / FLARE23 / RSNA** 其中一格或全部。结果在 `pretrain_lists_inventory/gcs_samples/` 与 `gcs_inventory_partial.json`。

- **BraTS**：contrast 配对见阶段三；可把 `FULL_SCAN_BRATS=True` 生成 `brats_preprocessed_npy_all_npy.txt`（很慢）。
- **FLARE23 / RSNA**：spatial；全量列表亦可复制到 `/tmp/all_flare23_gcs_npy.txt` 等在阶段三并入。

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
# BraTS 桶（contrast 源）
FULL_SCAN_BRATS = False  # True 递归全桶 .npy，耗时很长
gcs_scan_one_bucket("brats_preprocessed_npy", full_scan=FULL_SCAN_BRATS, sample_cap=50)

In [ ]:
# FLARE23 桶（spatial）
FULL_SCAN_FLARE23 = False
gcs_scan_one_bucket("flare23", full_scan=FULL_SCAN_FLARE23, sample_cap=50)

In [ ]:
# RSNA-2022-PED npy2（spatial）
FULL_SCAN_RSNA = False
gcs_scan_one_bucket("rsna_2022_ped_npy2", full_scan=FULL_SCAN_RSNA, sample_cap=50)

## 阶段三 — **合并** spatial / semantic / contrast

在所有需要的 **单数据集扫描** 完成后再运行下一格。

In [ ]:
random.seed(42)

# ---------- 合并 Drive spatial ----------
spatial_merge: list[str] = []
seen_sp = set()
per_dir = LISTS_OUT / "per_dataset"
if per_dir.is_dir():
    for fp in sorted(per_dir.glob("*_spatial.txt")):
        if fp.name.startswith("DeepLesion"):
            continue
        for line in open(fp, encoding="utf-8"):
            s = line.strip()
            if not s or s in seen_sp:
                continue
            seen_sp.add(s)
            spatial_merge.append(s)
spatial_merge.sort()
write_lines(LISTS_OUT / "spatial_merged_drive.txt", spatial_merge)
print("spatial_merged_drive.txt lines:", len(spatial_merge))

# ---------- 可选：/tmp 下 GCS 全量路径并入 ----------
tmp_flare23 = Path("/tmp/all_flare23_gcs_npy.txt")
flare_paths: list[str] = []
if tmp_flare23.is_file() and tmp_flare23.stat().st_size > 0:
    flare_paths = [x.strip() for x in open(tmp_flare23, encoding="utf-8") if x.strip().endswith(".npy")]
write_lines(LISTS_OUT / "spatial_flare23_gcs.txt", sorted(flare_paths))
print("spatial_flare23_gcs.txt lines:", len(flare_paths), "(来自 /tmp 或为空)")

tmp_rsna = Path("/tmp/all_rsna_ped_npy.txt")
rsna_paths: list[str] = []
if tmp_rsna.is_file() and tmp_rsna.stat().st_size > 0:
    rsna_paths = [x.strip() for x in open(tmp_rsna, encoding="utf-8") if x.strip().endswith(".npy")]
write_lines(LISTS_OUT / "spatial_rsna_ped_gcs.txt", sorted(rsna_paths))
print("spatial_rsna_ped_gcs.txt lines:", len(rsna_paths), "(来自 /tmp 或为空)")

spatial_full = list(spatial_merge)
seen_full = set(spatial_merge)
for p in flare_paths + rsna_paths:
    if p and p not in seen_full:
        seen_full.add(p)
        spatial_full.append(p)
spatial_full.sort()
write_lines(LISTS_OUT / "spatial_merged_full.txt", spatial_full)
print("spatial_merged_full.txt lines:", len(spatial_full))

# ---------- DeepLesion semantic ----------
deeplesion_root = find_under_pretrain("DeepLesion")
dl_npy = deeplesion_root / "npy" if deeplesion_root else None
semantic_lines: list[str] = []
if dl_npy and dl_npy.is_dir():
    grouped = defaultdict(list)
    for fn in tqdm(os.listdir(dl_npy), desc="DeepLesion"):
        if not fn.endswith(".npy"):
            continue
        full_path = str(dl_npy / fn)
        try:
            data = np.load(full_path, mmap_mode="r")
            if data.size == 0:
                continue
        except Exception:
            continue
        stem = fn.replace(".npy", "")
        lesion_type = stem.split("_")[-1] if "_" in stem else "0"
        grouped[lesion_type].append(full_path)
    for _, paths in grouped.items():
        if len(paths) < 4:
            continue
        for _ in range(len(paths)):
            semantic_lines.append(",".join(random.sample(paths, 4)))
    write_lines(LISTS_OUT / "semantic_deeplesion.txt", semantic_lines)
    print("semantic_deeplesion.txt lines:", len(semantic_lines))
else:
    write_lines(LISTS_OUT / "semantic_deeplesion.txt", [])
    print("DeepLesion/npy 不存在 -> 空 semantic_deeplesion.txt")

# ---------- BraTS contrast pair4 ----------
BRATS_GCS = GCS_DATASETS["brats_preprocessed_npy"]
tmp_brats = Path("/tmp/all_brats_gcs_npy.txt")
gcs_brats_all = LISTS_OUT / "gcs_samples" / "brats_preprocessed_npy_all_npy.txt"
brats_list_file = tmp_brats if tmp_brats.is_file() and tmp_brats.stat().st_size > 0 else gcs_brats_all
print("BraTS 列表源:", brats_list_file if brats_list_file.is_file() else "无")
if brats_list_file.is_file() and brats_list_file.stat().st_size > 0:
    all_files = [x.strip() for x in open(brats_list_file, encoding="utf-8") if x.strip().endswith(".npy")]
    all_set = set(all_files)
    t1n_files = [f for f in all_files if ".t1n.npy" in f]
    pair4: list[str] = []
    for t1n in tqdm(t1n_files, desc="BraTS pair4"):
        base = t1n.replace(".t1n.npy", "")
        req = [f"{base}.t1n.npy", f"{base}.t1c.npy", f"{base}.t2w.npy", f"{base}.t2f.npy"]
        if all(x in all_set for x in req):
            pair4.append(",".join(req))
    write_lines(LISTS_OUT / "contrast_brats_gcs_pair4.txt", pair4)
    print("contrast_brats_gcs_pair4.txt lines:", len(pair4))
else:
    write_lines(LISTS_OUT / "contrast_brats_gcs_pair4.txt", [])
    print("无 BraTS 全量列表；请先 FULL_SCAN_BRATS=True 跑 BraTS 格，或:")
    print("  gsutil ls -r '" + BRATS_GCS + "/**/*.npy' > /tmp/all_brats_gcs_npy.txt")

print("\n合并完成。spatial:", LISTS_OUT / "spatial_merged_full.txt")
print("contrast:", LISTS_OUT / "contrast_brats_gcs_pair4.txt")
print("semantic:", LISTS_OUT / "semantic_deeplesion.txt")